# Evidence checks for conditional next-SFT protocol
No private content. Likelihood is not generated correctness; order-swapped selected audits are not whole-benchmark accuracy.


In [ ]:
import json
from pathlib import Path
p=Path('docs') if Path('docs').is_dir() else Path('.')
models={k:json.loads((p/f'targeted_pilot_nll_{k}_20260908.safe.json').read_text(encoding='utf-8')) for k in ('baseline','step5','step10')}
assert len({m['input_sha256'] for m in models.values()})==1
for cohort,n in [('opportunity',64),('maintenance',16),('development',39)]:
    rr=[models[k]['cohorts'][cohort] for k in models]
    assert all(r['items']==n for r in rr)
    assert len({r['content_tokens'] for r in rr})==1
    assert rr[0]['content_nll']>rr[1]['content_nll']>rr[2]['content_nll']
b=models['baseline']['cohorts']['opportunity']; s=models['step10']['cohorts']['opportunity']
eos_gain=64*(b['mean_eos_nll']-s['mean_eos_nll'])
content_gain=b['content_tokens']*(b['content_nll']-s['content_nll'])
assert 0.08<eos_gain/(eos_gain+content_gain)<0.09
j=json.loads((p/'targeted_pilot_judge_audit_20260908.safe.json').read_text(encoding='utf-8'))
assert j['calls']==56 and j['statuses']=={'valid':56}
assert sum(g['statuses']['order_disagreement_pair'] for g in j['groups'].values())==7
assert all(sum(g['statuses'].values())==14 for g in j['groups'].values())
assert j['usage_tokens']==258247
print('PASS: exact inputs, NLL cohorts, EOS decomposition and judge order disagreement.')
